In [ ]:
%pip install --ignore-installed -r path/to/requirements_container.txt

In [ ]:
from __future__ import annotations

import csv
import json
import math
import os
import re
import socket
import subprocess
import sys
import textwrap
import threading
import time
import urllib.error
import urllib.request
from dataclasses import asdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any
from typing import Sequence

import pandas as pd
from openai import OpenAI
from transformers import AutoTokenizer

In [ ]:
HF_TOKEN = "your_hf_token"
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

In [ ]:
class CFG:
    
    project_root = Path(
        os.environ.get(
            "AIMO_PROJECT_ROOT",
            str(Path.cwd()),
        )
    ).resolve()

    requirements_path = Path(
        os.environ.get(
            "AIMO_REQUIREMENTS_PATH",
            "path/to/requirements_container.txt",
        )
    ).expanduser().resolve()

    eval_parquet_path = Path(
        os.environ.get(
            "AIMO_EVAL_PARQUET_PATH",
            "path/to/aimo_proof_eval.parquet",
        )
    ).expanduser().resolve()

    venv_path = Path(
        os.environ.get(
            "AIMO_VENV_PATH",
            str(project_root / "venv-container"),
        )
    ).resolve()

    venv_bin_path = venv_path / "bin"

    configured_venv_python_path = Path(
        os.environ.get(
            "AIMO_VENV_PYTHON_PATH",
            str(venv_bin_path / "python"),
        )
    ).resolve()

    if configured_venv_python_path.exists() and configured_venv_python_path.parent == venv_bin_path:
        venv_python_path = configured_venv_python_path

    elif (venv_bin_path / "python").exists():
        venv_python_path = (venv_bin_path / "python").resolve()

    else:
        venv_python_path = Path(sys.executable).resolve()

    model_path = os.environ.get(
        "AIMO_MODEL_PATH",
        "allenai/Olmo-3.1-32B-Think",
    )

    served_model_name = os.environ.get(
        "AIMO_SERVED_MODEL_NAME",
        "OLMo-3.1-32B-Think",
    )

    trust_remote_code = (
        os.environ.get("AIMO_TRUST_REMOTE_CODE", "true").strip().lower()
        in {"1", "true", "yes", "on"}
    )

    host = os.environ.get(
        "AIMO_HOST",
        "127.0.0.1",
    )

    server_port = int(
        os.environ.get(
            "AIMO_PORT",
            "8000",
        )
    )

    base_url = f"http://{host}:{server_port}/v1"
    api_key = "sk-local"

    launch_server = (
        os.environ.get("AIMO_LAUNCH_SERVER", "true").strip().lower()
        in {"1", "true", "yes", "on"}
    )

    reuse_server = (
        os.environ.get("AIMO_REUSE_SERVER", "false").strip().lower()
        in {"1", "true", "yes", "on"}
    )

    output_root = Path(
        os.environ.get(
            "AIMO_OUTPUT_ROOT",
            str(project_root / "output" / "runpod"),
        )
    ).resolve()

    output_path = Path(
        os.environ.get(
            "AIMO_OUTPUT_TXT",
            str(output_root / "aimo_proof_outputs.txt"),
        )
    ).resolve()

    log_path = Path(
        os.environ.get(
            "AIMO_LOG_PATH",
            str(output_root / "aimo_notebook_status.json"),
        )
    ).resolve()

    cache_root = Path(
        os.environ.get(
            "AIMO_CACHE_ROOT",
            str(output_root / "cache"),
        )
    ).resolve()

    hf_home = Path(
        os.environ.get(
            "HF_HOME",
            str(cache_root / "huggingface"),
        )
    )

    transformers_cache = Path(
        os.environ.get(
            "TRANSFORMERS_CACHE",
            str(hf_home / "transformers"),
        )
    )

    huggingface_hub_cache = Path(
        os.environ.get(
            "HUGGINGFACE_HUB_CACHE",
            str(hf_home / "hub"),
        )
    )

    pip_cache = Path(
        os.environ.get(
            "PIP_CACHE_DIR",
            str(cache_root / "pip"),
        )
    )

    torch_home = Path(
        os.environ.get(
            "TORCH_HOME",
            str(cache_root / "torch"),
        )
    )

    triton_cache = Path(
        os.environ.get(
            "TRITON_CACHE_DIR",
            str(cache_root / "triton"),
        )
    )

    vllm_cache = Path(
        os.environ.get(
            "VLLM_CACHE_ROOT",
            str(cache_root / "vllm"),
        )
    )

    seed = int(
        os.environ.get(
            "AIMO_SEED",
            "42",
        )
    )

    quick_run = (
        os.environ.get("AIMO_QUICK_RUN", "true").strip().lower()
        in {"1", "true", "yes", "on"}
    )

    quick_run_problem_count = int(
        os.environ.get(
            "AIMO_QUICK_RUN_PROBLEM_COUNT",
            "2",
        )
    )

    quick_run_seed = int(
        os.environ.get(
            "AIMO_QUICK_RUN_SEED",
            str(seed),
        )
    )

    problem_limit = int(
        os.environ.get(
            "AIMO_PROBLEM_LIMIT",
            "0",
        )
    )

    max_model_len = int(
        os.environ.get(
            "AIMO_MAX_MODEL_LEN",
            "65536",
        )
    )

    max_generation_tokens = int(
        os.environ.get(
            "AIMO_MAX_GENERATION_TOKENS",
            "57344",
        )
    )

    minimum_available_tokens = int(
        os.environ.get(
            "AIMO_MINIMUM_AVAILABLE_TOKENS",
            "512",
        )
    )

    tensor_parallel_size = int(
        os.environ.get(
            "AIMO_TENSOR_PARALLEL_SIZE",
            "1",
        )
    )

    max_num_seqs = int(
        os.environ.get(
            "AIMO_MAX_NUM_SEQS",
            "8",
        )
    )

    max_num_batched_tokens = int(
        os.environ.get(
            "AIMO_MAX_NUM_BATCHED_TOKENS",
            "8192",
        )
    )

    max_logprobs = int(
        os.environ.get(
            "AIMO_MAX_LOGPROBS",
            "0",
        )
    )

    request_logprobs = int(
        os.environ.get(
            "AIMO_REQUEST_LOGPROBS",
            "0",
        )
    )

    stream_interval = int(
        os.environ.get(
            "AIMO_STREAM_INTERVAL",
            "256",
        )
    )

    enable_async_scheduling = (
        os.environ.get("AIMO_ENABLE_ASYNC_SCHEDULING", "true").strip().lower()
        in {"1", "true", "yes", "on"}
    )

    enable_prefix_caching = (
        os.environ.get("AIMO_ENABLE_PREFIX_CACHING", "true").strip().lower()
        in {"1", "true", "yes", "on"}
    )

    enable_chunked_prefill = (
        os.environ.get("AIMO_ENABLE_CHUNKED_PREFILL", "true").strip().lower()
        in {"1", "true", "yes", "on"}
    )

    disable_log_stats = (
        os.environ.get("AIMO_DISABLE_LOG_STATS", "true").strip().lower()
        in {"1", "true", "yes", "on"}
    )

    gpu_memory_utilization = float(
        os.environ.get(
            "AIMO_GPU_MEMORY_UTILIZATION",
            "0.95",
        )
    )

    compilation_config = {
        "cudagraph_capture_sizes": [
            1,
            2,
            4,
            8,
        ],
        "cudagraph_specialize_lora": False,
        "max_cudagraph_capture_size": 8,
    }

    temperature = float(
        os.environ.get(
            "AIMO_TEMPERATURE",
            "0.6",
        )
    )

    top_p = float(
        os.environ.get(
            "AIMO_TOP_P",
            "0.95",
        )
    )

    top_k = int(
        os.environ.get(
            "AIMO_TOP_K",
            "-1",
        )
    )

    min_p = float(
        os.environ.get(
            "AIMO_MIN_P",
            "0.0",
        )
    )

    presence_penalty = float(
        os.environ.get(
            "AIMO_PRESENCE_PENALTY",
            "0.0",
        )
    )

    repetition_penalty = float(
        os.environ.get(
            "AIMO_REPETITION_PENALTY",
            "1.0",
        )
    )

    stop_strings = [
        "<|im_end|>",
    ]

    request_timeout = float(
        os.environ.get(
            "AIMO_REQUEST_TIMEOUT_SECONDS",
            "3420",
        )
    )

    output_ping_interval = float(
        os.environ.get(
            "AIMO_OUTPUT_PING_SECONDS",
            "15",
        )
    )

    output_pulse_enabled = (
        os.environ.get("AIMO_ENABLE_OUTPUT_PULSE", "true").strip().lower()
        in {"1", "true", "yes", "on"}
    )

    cuda_preflight_enabled = (
        os.environ.get("AIMO_ENABLE_CUDA_PREFLIGHT", "true").strip().lower()
        in {"1", "true", "yes", "on"}
    )

    server_timeout = float(
        os.environ.get(
            "AIMO_SERVER_START_TIMEOUT_SECONDS",
            "900",
        )
    )

    pass_timeout = float(
        os.environ.get(
            "AIMO_PASS_TIMEOUT_SECONDS",
            "3300",
        )
    )

    vllm_chat_template_content_format = os.environ.get(
        "AIMO_VLLM_CHAT_TEMPLATE_CONTENT_FORMAT",
        "string",
    ).strip()

    problem_columns = (
        "problem",
        "problem_markdown",
        "statement",
        "prompt",
        "question",
    )

    reference_columns = (
        "solution",
        "solution_markdown",
        "reference_solution",
    )

    first_pass_system_prompt = textwrap.dedent(
        """
        You are one of the most capable contestants selected for this year's International Mathematical Olympiad, seated for the Contest and preparing a solution to be read by experienced olympiad graders. The work should have the form of a contest submission: a coherent argument in standard mathematical prose, with definitions introduced before they are used, notation kept stable, lemmas justified, cases separated cleanly, and the conclusion following without gaps or contradictions.

        The final answer must prove every step of the solution: every reduction, identity, inequality, case split, finiteness claim, monotonicity claim, divisibility claim, extremal claim, and appeal to a known theorem must be justified in the text before it is used. When the final solution is ready, close any <think> section and then write only the solution itself: no drafting notes, and no discussion of private reasoning.
        """
    ).strip()

    first_pass_prompt = textwrap.dedent(
        """
        Problem:
        {problem}

        A complete solution is required. The submitted answer should read like a contest solution: text-only, self-contained, step by step, and consistent in notation and cases. Prove every nontrivial assertion before relying on it; do not use phrases such as clearly, by computation, it is known, it follows, or this pattern continues unless the reason is supplied in the proof. Close any <think> section before writing the final submitted solution.
        """
    ).strip()

    vllm_environment = {
        "VLLM_DO_NOT_TRACK": "1",
        "VLLM_NO_USAGE_STATS": "1",
        "TRANSFORMERS_NO_TF": "1",
        "TRANSFORMERS_NO_FLAX": "1",
        "TOKENIZERS_PARALLELISM": "false",
        "PYTORCH_ALLOC_CONF": "expandable_segments:True",
        "TORCH_COMPILE_DISABLE": "1",
        "TORCHDYNAMO_DISABLE": "1",
        "TORCHINDUCTOR_DISABLE": "1",
        "AIMO_DISABLE_TORCH_COMPILE": "1",
        "VLLM_LOG_STATS_INTERVAL": "60",
        "VLLM_USE_V2_MODEL_RUNNER": "1",
    }

In [ ]:
@dataclass(frozen=True)
class AIMOProblemRecord:
    
    order_index: int
    id: str
    problem: str
    reference_solution: str
    metadata: dict[str, str]


@dataclass(frozen=True)
class AIMOChatMessage:
    
    role: str
    content: str


@dataclass(frozen=True)
class AIMOChatResponse:
    
    content: str


@dataclass(frozen=True)
class AIMOPassResult:
    
    name: str
    final_text: str
    elapsed_seconds: float
    input_tokens: int
    output_tokens: int


@dataclass(frozen=True)
class AIMOProblemResult:
    
    record: AIMOProblemRecord
    passes: list[AIMOPassResult]


class AIMOOutputPulse:
    
    def __init__(self, interval_seconds: float, enabled: bool) -> None:
        
        self.interval_seconds = interval_seconds
        self.enabled = enabled
        self.stop_event = threading.Event()
        self.thread = threading.Thread(target=self.run, daemon=True)
        self.dots_on_line = 0

    def __enter__(self) -> AIMOOutputPulse:
        
        if self.enabled and self.interval_seconds > 0:
            self.thread.start()

        return self

    def __exit__(self, exception_type: object, exception: object, traceback: object) -> None:
        
        self.stop()

    def run(self) -> None:
        
        while not self.stop_event.wait(self.interval_seconds):
            print(".", end="", flush=True)
            self.dots_on_line += 1

            if self.dots_on_line >= 10:
                print("", flush=True)
                self.dots_on_line = 0

    def stop(self) -> None:
        
        self.stop_event.set()

        if self.thread.is_alive():
            self.thread.join(timeout=1.0)

        if self.enabled:
            print("", flush=True)

In [ ]:
class AIMOEnvironment:
    
    def __init__(self, cfg: CFG) -> None:
        
        self.cfg = cfg

    def configure(self) -> None:
        
        for path in self.cache_paths():
            path.mkdir(parents=True, exist_ok=True)

        self.cfg.output_path.parent.mkdir(parents=True, exist_ok=True)
        self.cfg.log_path.parent.mkdir(parents=True, exist_ok=True)
        self.export_paths()
        self.validate_paths()

    def export_paths(self) -> None:
        
        os.environ["AIMO_PROJECT_ROOT"] = str(self.cfg.project_root)
        os.environ["AIMO_OUTPUT_ROOT"] = str(self.cfg.output_root)
        os.environ["AIMO_REQUIREMENTS_PATH"] = str(self.cfg.requirements_path)
        os.environ["AIMO_EVAL_PARQUET_PATH"] = str(self.cfg.eval_parquet_path)
        os.environ["AIMO_VENV_PATH"] = str(self.cfg.venv_path)
        os.environ["AIMO_VENV_PYTHON_PATH"] = str(self.cfg.venv_python_path)
        os.environ["HF_HOME"] = str(self.cfg.hf_home)
        os.environ["TRANSFORMERS_CACHE"] = str(self.cfg.transformers_cache)
        os.environ["HUGGINGFACE_HUB_CACHE"] = str(self.cfg.huggingface_hub_cache)
        os.environ["PIP_CACHE_DIR"] = str(self.cfg.pip_cache)
        os.environ["TORCH_HOME"] = str(self.cfg.torch_home)
        os.environ["TRITON_CACHE_DIR"] = str(self.cfg.triton_cache)
        os.environ["VLLM_CACHE_ROOT"] = str(self.cfg.vllm_cache)
        os.environ.update(self.cfg.vllm_environment)

        if self.cfg.venv_python_path.parent == self.cfg.venv_bin_path:
            current_path = os.environ.get("PATH", "")
            os.environ["VIRTUAL_ENV"] = str(self.cfg.venv_path)
            os.environ["PATH"] = os.pathsep.join([
                str(self.cfg.venv_bin_path),
                current_path,
            ])

    def validate_paths(self) -> None:
        
        if not self.cfg.requirements_path.exists():
            raise FileNotFoundError(f"requirements file does not exist: {self.cfg.requirements_path}")

        if not self.cfg.eval_parquet_path.exists():
            raise FileNotFoundError(f"eval parquet does not exist: {self.cfg.eval_parquet_path}")

        if not self.cfg.eval_parquet_path.is_file():
            raise FileNotFoundError(f"eval parquet is not a file: {self.cfg.eval_parquet_path}")

        if not self.cfg.venv_python_path.exists():
            raise FileNotFoundError(f"venv python does not exist: {self.cfg.venv_python_path}")

    def cache_paths(self) -> list[Path]:
        
        return [
            self.cfg.cache_root,
            self.cfg.hf_home,
            self.cfg.transformers_cache,
            self.cfg.huggingface_hub_cache,
            self.cfg.pip_cache,
            self.cfg.torch_home,
            self.cfg.triton_cache,
            self.cfg.vllm_cache,
        ]


class AIMOInputReader:
    
    def __init__(self, cfg: CFG) -> None:
        
        self.cfg = cfg

    def read_records(self) -> list[AIMOProblemRecord]:
        
        dataset_path = self.find_dataset_path()
        dataframe = pd.read_parquet(dataset_path)

        if self.cfg.quick_run:
            dataframe = self.quick_run_dataframe(dataframe)

        elif self.cfg.problem_limit > 0:
            dataframe = dataframe.head(self.cfg.problem_limit)

        records = [
            self.record_from_row(row_payload, order_index)
            for order_index, row_payload in enumerate(dataframe.to_dict(orient="records"))
        ]

        return records

    def quick_run_dataframe(self, dataframe: pd.DataFrame) -> pd.DataFrame:
        
        sample_size = min(len(dataframe), max(0, self.cfg.quick_run_problem_count))

        if sample_size <= 0:
            return dataframe.head(0)

        sampled_dataframe = dataframe.sample(
            n=sample_size,
            random_state=self.cfg.quick_run_seed,
        ).sort_index()

        return sampled_dataframe

    def find_dataset_path(self) -> Path:
        
        path = self.cfg.eval_parquet_path

        if not path.exists():
            raise FileNotFoundError(f"eval parquet does not exist: {path}")

        if not path.is_file():
            raise FileNotFoundError(f"eval parquet is not a file: {path}")

        return path

    def record_from_row(self, row_payload: dict[str, object], order_index: int) -> AIMOProblemRecord:
        
        row = {
            str(key): value
            for key, value in row_payload.items()
        }
        problem = self.select_first(row, self.cfg.problem_columns)
        reference_solution = self.select_first(row, self.cfg.reference_columns)
        record_id = self.clean_value(row.get("id") or row.get("problem_id") or order_index)
        ignored_columns = {"id", "problem_id", *self.cfg.problem_columns, *self.cfg.reference_columns}
        metadata = {
            key: self.clean_value(value)
            for key, value in row.items()
            if key not in ignored_columns
        }

        if not problem.strip():
            raise ValueError(f"Problem text is empty for row {order_index}.")

        return AIMOProblemRecord(
            order_index=order_index,
            id=record_id,
            problem=problem,
            reference_solution=reference_solution,
            metadata=metadata,
        )

    def select_first(self, row: dict[str, object], columns: Sequence[str]) -> str:
        
        for column in columns:
            if column in row:
                value = self.clean_value(row[column]).strip()

                if value:
                    return value

        return ""

    def clean_value(self, value: object) -> str:
        
        if value is None:
            return ""

        try:
            if pd.isna(value):
                return ""
            
        except (TypeError, ValueError):
            pass

        return str(value)


class AIMOOutputWriter:
    
    def __init__(self, cfg: CFG) -> None:
        
        self.cfg = cfg

    def write_results(self, results: list[AIMOProblemResult]) -> None:
        
        blocks = [
            self.format_problem_result(result)
            for result in results
        ]

        self.cfg.output_path.write_text(
            "\n\n".join(blocks).strip() + "\n",
            encoding="utf-8",
        )

        self.write_status(results)

    def write_status(self, results: list[AIMOProblemResult]) -> None:
        
        payload = {
            "problem_count": len(results),
            "output_path": str(self.cfg.output_path),
            "totals": self.metrics_totals(results),
            "problems": [
                {
                    "id": result.record.id,
                    "passes": [
                        {
                            "name": pass_result.name,
                            "elapsed_seconds": pass_result.elapsed_seconds,
                            "input_tokens": pass_result.input_tokens,
                            "generated_tokens": pass_result.output_tokens,
                            "total_tokens": pass_result.input_tokens + pass_result.output_tokens,
                        }
                        for pass_result in result.passes
                    ],
                }
                for result in results
            ],
        }
        
        self.cfg.log_path.write_text(
            json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
        )

    def metrics_dataframe(self, results: list[AIMOProblemResult]) -> pd.DataFrame:
        
        rows = []

        for result in results:
            for pass_index, pass_result in enumerate(result.passes, start=1):
                rows.append({
                    "problem_id": result.record.id,
                    "pass_index": pass_index,
                    "pass_name": pass_result.name,
                    "input_tokens": pass_result.input_tokens,
                    "generated_tokens": pass_result.output_tokens,
                    "total_tokens": pass_result.input_tokens + pass_result.output_tokens,
                    "elapsed_seconds": round(pass_result.elapsed_seconds, 2),
                })

        totals = self.metrics_totals(results)

        if rows:
            rows.append({
                "problem_id": "TOTAL",
                "pass_index": 0,
                "pass_name": "all",
                "input_tokens": totals["input_tokens"],
                "generated_tokens": totals["generated_tokens"],
                "total_tokens": totals["total_tokens"],
                "elapsed_seconds": round(totals["elapsed_seconds"], 2),
            })

        return pd.DataFrame(rows)

    def metrics_totals(self, results: list[AIMOProblemResult]) -> dict[str, int | float]:
        
        passes = [
            pass_result
            for result in results
            for pass_result in result.passes
        ]

        input_tokens = sum(pass_result.input_tokens for pass_result in passes)
        generated_tokens = sum(pass_result.output_tokens for pass_result in passes)

        return {
            "input_tokens": input_tokens,
            "generated_tokens": generated_tokens,
            "total_tokens": input_tokens + generated_tokens,
            "elapsed_seconds": sum(pass_result.elapsed_seconds for pass_result in passes),
        }

    def format_problem_result(self, result: AIMOProblemResult) -> str:
        
        sections = [
            "=" * 80,
            f"Problem ID: {result.record.id}",
            "",
            "Problem:",
            result.record.problem.strip(),
        ]

        for pass_index, pass_result in enumerate(result.passes, start=1):
            sections.extend([
                "",
                f"{pass_result.name.title()} Final Channel:",
                pass_result.final_text.strip() or "No final channel was produced.",
            ])

        return "\n".join(sections)

In [ ]:
class AIMOChatMLTemplate:
    
    think_pattern = re.compile(r"<think>.*?</think>", re.DOTALL)
    im_tag_pattern = re.compile(r"<\|im_start\|>.*?|<\|im_end\|>", re.DOTALL)

    def __init__(self, tokenizer: Any) -> None:
        
        self.tokenizer = tokenizer

    def render_initial_prompt(self, messages: Sequence[AIMOChatMessage]) -> str:
        
        return self.render_messages(
            messages=messages,
            add_generation_prompt=True,
        )

    def render_messages(
        self,
        messages: Sequence[AIMOChatMessage],
        add_generation_prompt: bool,
    ) -> str:
        
        rendered_messages = [
            self.message_payload(message)
            for message in messages
        ]

        return str(
            self.tokenizer.apply_chat_template(
                rendered_messages,
                tokenize=False,
                add_generation_prompt=add_generation_prompt,
            )
        )

    def message_payload(self, message: AIMOChatMessage) -> dict[str, Any]:
        
        return {
            "role": message.role,
            "content": self.escape_content(message.content),
        }

    def final_channel(self, text: str) -> str:
        
        if "</think>" in text:
            final_text = text.split("</think>")[-1]

        elif "<think>" in text:
            return "No final answer was produced."
        
        else:
            final_text = text

        final_text = self.think_pattern.sub("", final_text)
        final_text = self.im_tag_pattern.sub("", final_text)
        final_text = final_text.replace("<think>", "")
        final_text = final_text.replace("</think>", "")

        return "\n".join(
            line.rstrip()
            for line in final_text.splitlines()
        ).strip()

    def escape_content(self, content: str) -> str:
        
        return (
            str(content)
            .replace("<|im_start|>", "<|escaped_im_start|>")
            .replace("<|im_end|>", "<|escaped_im_end|>")
        )

In [ ]:
class AIMOServer:
    
    def __init__(self, cfg: CFG) -> None:
        
        self.cfg = cfg
        self.process = None
        self.stdout_file = None
        self.stderr_file = None
        self.stdout_path = self.cfg.output_root / "vllm_stdout.log"
        self.stderr_path = self.cfg.output_root / "vllm_stderr.log"
        self.command_path = self.cfg.output_root / "vllm_command.json"

    def __enter__(self) -> AIMOServer:
        
        self.start()

        return self

    def __exit__(self, exception_type: object, exception: object, traceback: object) -> None:
        
        self.stop()

    def start(self) -> None:
        
        if self.cfg.reuse_server:
            self.wait_until_ready()
            return

        if not self.cfg.launch_server:
            return

        if not self.port_available(self.cfg.host, self.cfg.server_port):
            raise RuntimeError(f"Port is already in use: {self.cfg.host}:{self.cfg.server_port}")

        self.validate_cuda_runtime()

        command = self.build_command()
        self.command_path.write_text(
            json.dumps({"command": command, "health_url": self.health_url()}, indent=2) + "\n",
            encoding="utf-8",
        )
        self.stdout_file = self.stdout_path.open("w", encoding="utf-8")
        self.stderr_file = self.stderr_path.open("w", encoding="utf-8")
        self.process = subprocess.Popen(
            command,
            stdout=self.stdout_file,
            stderr=self.stderr_file,
            text=True,
            env=self.server_environment(),
        )
        self.wait_until_ready()

    def stop(self) -> None:
        
        if self.process is not None and self.process.poll() is None:
            self.process.terminate()

            try:
                self.process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                self.process.kill()
                self.process.wait(timeout=30)

        for file_object in [self.stdout_file, self.stderr_file]:
            if file_object is not None:
                file_object.close()

    def python_executable(self) -> Path:
        
        if not self.cfg.venv_python_path.exists():
            raise FileNotFoundError(f"vLLM server python does not exist: {self.cfg.venv_python_path}")

        return self.cfg.venv_python_path

    def validate_cuda_runtime(self) -> None:
        
        if not self.cfg.cuda_preflight_enabled:
            return

        script = (
            "import torch\n"
            "print(f'torch={torch.__version__}')\n"
            "print(f'torch_cuda={torch.version.cuda}')\n"
            "print(f'cuda_available={torch.cuda.is_available()}')\n"
            "torch.cuda.init()\n"
            "print(f'cuda_device_count={torch.cuda.device_count()}')\n"
            "print(f'cuda_device_name={torch.cuda.get_device_name(0)}')\n"
        )

        try:
            subprocess.run(
                [str(self.python_executable()), "-c", script],
                capture_output=True,
                check=True,
                env=self.server_environment(),
                text=True,
                timeout=60,
            )

        except subprocess.CalledProcessError as exception:
            raise RuntimeError(
                "CUDA runtime preflight failed before vLLM launch.\n"
                "stdout:\n"
                f"{exception.stdout.strip()}\n"
                "stderr:\n"
                f"{exception.stderr.strip()}"
            ) from exception
        
        except subprocess.TimeoutExpired as exception:
            raise RuntimeError(
                "CUDA runtime preflight timed out before vLLM launch.\n"
                f"stdout:\n{str(exception.stdout or '').strip()}\n"
                f"stderr:\n{str(exception.stderr or '').strip()}"
            ) from exception

    def build_command(self) -> list[str]:
        
        command = [
            str(self.python_executable()),
            "-m",
            "vllm.entrypoints.openai.api_server",
            "--seed",
            str(self.cfg.seed),
            "--model",
            self.cfg.model_path,
            "--served-model-name",
            self.cfg.served_model_name,
            "--host",
            self.cfg.host,
            "--port",
            str(self.cfg.server_port),
            "--tensor-parallel-size",
            str(self.cfg.tensor_parallel_size),
            "--gpu-memory-utilization",
            str(self.cfg.gpu_memory_utilization),
            "--dtype",
            "auto",
            "--kv-cache-dtype",
            "auto",
            "--load-format",
            "auto",
            "--max-model-len",
            str(self.cfg.max_model_len),
            "--max-num-seqs",
            str(self.cfg.max_num_seqs),
            "--performance-mode",
            "throughput",
            "--compilation-config",
            json.dumps(self.cfg.compilation_config),
            "--trust-remote-code",
            "--download-dir",
            str(self.cfg.huggingface_hub_cache),
            "--generation-config",
            "vllm",
        ]

        if self.cfg.max_logprobs >= 0:
            command.extend([
                "--max-logprobs",
                str(self.cfg.max_logprobs),
            ])

        if self.cfg.stream_interval > 0:
            command.extend([
                "--stream-interval",
                str(self.cfg.stream_interval),
            ])

        if self.cfg.enable_prefix_caching:
            command.append("--enable-prefix-caching")

        if self.cfg.enable_chunked_prefill:
            command.append("--enable-chunked-prefill")

        if self.cfg.disable_log_stats:
            command.append("--disable-log-stats")

        if self.cfg.enable_async_scheduling:
            command.append("--async-scheduling")

        if self.cfg.vllm_chat_template_content_format:
            command.extend([
                "--chat-template-content-format",
                self.cfg.vllm_chat_template_content_format,
            ])

        if self.cfg.max_num_batched_tokens > 0:
            command.extend([
                "--max-num-batched-tokens",
                str(self.cfg.max_num_batched_tokens),
            ])

        return command

    def wait_until_ready(self) -> None:
        
        deadline = time.time() + self.cfg.server_timeout
        last_error = ""

        while time.time() < deadline:
            if self.process is not None and self.process.poll() is not None:
                raise RuntimeError(self.exit_message())

            try:
                with urllib.request.urlopen(self.health_url(), timeout=5) as response:
                    if response.status == 200:
                        return
                    
            except (urllib.error.URLError, TimeoutError) as exception:
                last_error = str(exception)

            time.sleep(1)

        raise RuntimeError(f"vLLM server did not become ready: {last_error}")

    def health_url(self) -> str:
        
        host = "127.0.0.1" if self.cfg.host == "0.0.0.0" else self.cfg.host

        return f"http://{host}:{self.cfg.server_port}/health"

    def server_environment(self) -> dict[str, str]:
        
        startup_path = self.cfg.output_root / "python_startup"
        startup_path.mkdir(parents=True, exist_ok=True)
        sitecustomize_path = startup_path / "sitecustomize.py"
        sitecustomize_path.write_text(
            textwrap.dedent(
                """
                import os

                if os.environ.get("AIMO_DISABLE_TORCH_COMPILE", "1").lower() in {"1", "true", "yes", "on"}:
                    try:
                        import sys
                        import types

                        import torch

                        compile_fx_module = types.ModuleType("torch._inductor.compile_fx")

                        def _aimo_compile_fx(*args, **kwargs):

                            raise RuntimeError("torch._inductor.compile_fx is disabled for this notebook run")

                        def _aimo_compile_fx_inner(*args, **kwargs):

                            raise RuntimeError("torch._inductor.compile_fx_inner is disabled for this notebook run")

                        def _aimo_graph_returns_tuple(graph):

                            return True

                        compile_fx_module.compile_fx = _aimo_compile_fx
                        compile_fx_module.compile_fx_inner = _aimo_compile_fx_inner
                        compile_fx_module.graph_returns_tuple = _aimo_graph_returns_tuple
                        sys.modules["torch._inductor.compile_fx"] = compile_fx_module

                        if hasattr(torch, "_inductor"):
                            setattr(torch._inductor, "compile_fx", compile_fx_module)

                        def _aimo_no_compile(function=None, *args, **kwargs):

                            if function is None:
                                def _aimo_decorator(inner_function):

                                    return inner_function

                                return _aimo_decorator

                            return function

                        torch.compile = _aimo_no_compile
                    except Exception:
                        pass
                """
            ).strip() + "\n",
            encoding="utf-8",
        )
        environment = os.environ.copy()
        environment.update(self.cfg.vllm_environment)
        pythonpath = environment.get("PYTHONPATH", "")
        pythonpath_parts = [
            str(startup_path),
            *[
                part
                for part in pythonpath.split(os.pathsep)
                if part
            ],
        ]
        environment["PYTHONPATH"] = os.pathsep.join(dict.fromkeys(pythonpath_parts))

        return environment

    def exit_message(self) -> str:
        
        return_code = self.process.returncode if self.process is not None else "unknown"

        return "\n".join([
            "vLLM server exited before readiness.",
            f"return_code={return_code}",
            "stdout_tail:",
            self.tail(self.stdout_path),
            "stderr_tail:",
            self.tail(self.stderr_path),
        ])

    def tail(self, path: Path, max_bytes: int = 20000) -> str:
        
        if not path.exists():
            return f"missing log file: {path}"

        with path.open("rb") as input_file:
            input_file.seek(0, os.SEEK_END)
            size = input_file.tell()
            input_file.seek(max(0, size - max_bytes), os.SEEK_SET)
            payload = input_file.read()

        return payload.decode("utf-8", errors="replace")

    def port_available(self, host: str, port: int) -> bool:
        
        try:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as socket_object:
                socket_object.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
                socket_object.bind((host, port))

            return True
        except OSError:
            return False

In [ ]:
class AIMOInferenceClient:
    
    def __init__(self, cfg: CFG) -> None:
        
        self.cfg = cfg
        self.client = OpenAI(
            base_url=self.cfg.base_url,
            api_key=self.cfg.api_key,
            timeout=self.cfg.request_timeout,
        )
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.cfg.model_path,
            trust_remote_code=self.cfg.trust_remote_code,
        )

    def chat_completion(
        self,
        messages: list[dict[str, Any]],
        max_tokens: int,
    ) -> AIMOChatResponse:
        
        input_tokens = self.count_chat_tokens(messages)
        available_tokens = self.available_chat_output_tokens(input_tokens)
        resolved_max_tokens = self.resolve_output_token_count(available_tokens, max_tokens)
        payload = self.chat_sampling_payload(resolved_max_tokens)
        try:
            completion = self.client.chat.completions.create(
                messages=messages,
                **payload,
            )
        except Exception:
            return self.prompt_completion(
                messages=messages,
                max_tokens=max_tokens,
            )

        message = completion.choices[0].message

        return AIMOChatResponse(
            content=str(getattr(message, "content", "") or ""),
        )

    def prompt_completion(
        self,
        messages: list[dict[str, Any]],
        max_tokens: int,
        timeout_seconds: float | None = None,
    ) -> AIMOChatResponse:
        
        prompt = self.render_chat_prompt(messages)
        generated_text = self.stream_completion(
            prompt=prompt,
            max_tokens=max_tokens,
            stop_strings=self.cfg.stop_strings,
            timeout_seconds=timeout_seconds,
        )

        return AIMOChatResponse(content=generated_text)

    def render_chat_prompt(self, messages: list[dict[str, Any]]) -> str:
        
        try:
            return str(
                self.tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                )
            )
        except Exception:
            return self.render_manual_chat_prompt(messages)

    def render_manual_chat_prompt(self, messages: list[dict[str, Any]]) -> str:
        
        parts = []

        for message in messages:
            role = str(message.get("role", "user"))
            content = self.escape_chat_content(str(message.get("content", "")))
            parts.append(f"<|im_start|>{role}\n{content}<|im_end|>\n")

        parts.append("<|im_start|>assistant\n<think>")

        return "".join(parts)

    def escape_chat_content(self, content: str) -> str:
        
        return (
            content.replace("<|im_start|>", "<|escaped_im_start|>")
            .replace("<|im_end|>", "<|escaped_im_end|>")
        )

    def chat_sampling_payload(self, max_tokens: int) -> dict[str, Any]:
        
        payload = self.sampling_payload(
            max_tokens=max_tokens,
            stop_strings=self.cfg.stop_strings,
        )
        payload.pop("logprobs", None)

        return payload

    def stream_completion(
        self,
        prompt: str,
        max_tokens: int,
        stop_strings: Sequence[str] | None = None,
        timeout_seconds: float | None = None,
    ) -> str:
        
        resolved_max_tokens = self.resolve_output_tokens(prompt, max_tokens)
        payload = self.sampling_payload(
            max_tokens=resolved_max_tokens,
            stop_strings=stop_strings,
        )
        deadline = time.time() + timeout_seconds if timeout_seconds is not None and timeout_seconds > 0 else None
        stream = self.client.completions.create(
            prompt=prompt,
            stream=True,
            timeout=timeout_seconds if timeout_seconds is not None and timeout_seconds > 0 else self.cfg.request_timeout,
            **payload,
        )
        chunks = []

        try:
            for chunk in stream:
                if deadline is not None and time.time() >= deadline:
                    break

                text = str(chunk.choices[0].text or "")

                if text:
                    chunks.append(text)
        finally:
            stream.close()

        return "".join(chunks)

    def sampling_payload(
        self,
        max_tokens: int,
        stop_strings: Sequence[str] | None = None,
    ) -> dict[str, Any]:
        
        payload = {
            "model": self.cfg.served_model_name,
            "max_tokens": max_tokens,
            "temperature": self.cfg.temperature,
            "stop": list(stop_strings) if stop_strings is not None else self.cfg.stop_strings,
        }
        extra_body = {}

        if self.cfg.request_logprobs >= 0:
            payload["logprobs"] = self.cfg.request_logprobs

        if self.cfg.top_p > 0:
            payload["top_p"] = self.cfg.top_p

        if self.cfg.top_k > 0:
            extra_body["top_k"] = self.cfg.top_k

        if self.cfg.min_p > 0:
            extra_body["min_p"] = self.cfg.min_p

        if self.cfg.presence_penalty != 0.0:
            payload["presence_penalty"] = self.cfg.presence_penalty

        if self.cfg.repetition_penalty != 1.0:
            extra_body["repetition_penalty"] = self.cfg.repetition_penalty

        if extra_body:
            payload["extra_body"] = extra_body

        return payload

    def resolve_output_tokens(self, prompt: str, requested_max_tokens: int) -> int:
        
        available_tokens = self.available_output_tokens(prompt)

        return self.resolve_output_token_count(available_tokens, requested_max_tokens)

    def resolve_output_token_count(self, available_tokens: int, requested_max_tokens: int) -> int:
        
        resolved_max_tokens = min(
            available_tokens,
            self.cfg.max_generation_tokens,
            max(0, requested_max_tokens),
        )

        if resolved_max_tokens <= 0:
            raise ValueError("No output tokens are available for generation.")

        return resolved_max_tokens

    def available_output_tokens(self, prompt: str) -> int:
        
        input_tokens = self.count_tokens(prompt)
        available_tokens = self.cfg.max_model_len - input_tokens

        if available_tokens <= 0:
            raise ValueError(f"Prompt uses {input_tokens} tokens and exceeds max_model_len={self.cfg.max_model_len}.")

        return available_tokens

    def count_chat_tokens(self, messages: list[dict[str, Any]]) -> int:
        
        try:
            prompt = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        except Exception:
            prompt = self.render_manual_chat_prompt(messages)

        return self.count_tokens(str(prompt))

    def available_chat_output_tokens(self, input_tokens: int) -> int:
        
        available_tokens = self.cfg.max_model_len - input_tokens

        if available_tokens <= 0:
            raise ValueError(f"Prompt uses {input_tokens} tokens and exceeds max_model_len={self.cfg.max_model_len}.")

        return available_tokens

    def count_tokens(self, text: str) -> int:
        
        return len(self.tokenizer.encode(text, add_special_tokens=False))

In [ ]:
class AIMOPromptBuilder:
    
    def __init__(self, cfg: CFG) -> None:
        
        self.cfg = cfg

    def first_pass_messages(self, problem: str) -> list[AIMOChatMessage]:
        
        return self.messages(
            system_prompt=self.cfg.first_pass_system_prompt,
            user_prompt=self.cfg.first_pass_prompt.format(problem=problem.strip()),
        )

    def messages(self, system_prompt: str, user_prompt: str) -> list[AIMOChatMessage]:
        
        return [
            AIMOChatMessage(role="system", content=system_prompt),
            AIMOChatMessage(role="user", content=user_prompt.strip()),
        ]

In [ ]:
class AIMOSolver:
    
    def __init__(self, cfg: CFG, client: AIMOInferenceClient) -> None:
        
        self.cfg = cfg
        self.client = client
        self.template = AIMOChatMLTemplate(client.tokenizer)
        self.prompt_builder = AIMOPromptBuilder(cfg)

    def solve_problem(self, record: AIMOProblemRecord) -> AIMOProblemResult:
        
        first_pass = self.run_pass(
            name="solve",
            messages=self.prompt_builder.first_pass_messages(record.problem),
        )
        return AIMOProblemResult(
            record=record,
            passes=[first_pass],
        )

    def run_pass(
        self,
        name: str,
        messages: list[AIMOChatMessage],
    ) -> AIMOPassResult:
        
        started_at = time.time()
        transcript_messages = [
            self.template.message_payload(message)
            for message in messages
        ]
        implicit_think_opened = "think" in str(self.cfg.served_model_name).lower() or "think" in str(self.cfg.model_path).lower()
        raw_chunks = ["<think>"] if implicit_think_opened else []
        input_tokens = self.client.count_chat_tokens(transcript_messages)
        chat_response = self.generate_chat(
            messages=transcript_messages,
            output_tokens=0,
        )
        generated_text = chat_response.content
        output_tokens = self.client.count_tokens(generated_text)
        raw_chunks.append(generated_text)
        raw_text = "\n".join(raw_chunks)
        final_text = self.template.final_channel(raw_text)

        self.print_final_solution(name, final_text)

        return AIMOPassResult(
            name=name,
            final_text=final_text,
            elapsed_seconds=time.time() - started_at,
            input_tokens=input_tokens,
            output_tokens=output_tokens,
        )

    def generate_chat(
        self,
        messages: list[dict[str, Any]],
        output_tokens: int,
    ) -> AIMOChatResponse:
        
        return self.client.prompt_completion(
            messages=messages,
            max_tokens=self.remaining_generation_tokens(output_tokens),
            timeout_seconds=self.cfg.pass_timeout,
        )

    def print_final_solution(self, name: str, final_text: str) -> None:
        
        print("", flush=True)
        print(f"{name.title()} Final Channel:")
        print(final_text.strip() or "No final channel was produced.")
        print("", flush=True)

    def remaining_generation_tokens(self, output_tokens: int) -> int:
        
        return max(0, self.cfg.max_generation_tokens - output_tokens)

In [ ]:
class AIMORunner:
    
    def __init__(self, cfg: CFG) -> None:
        
        self.cfg = cfg
        self.environment = AIMOEnvironment(cfg)
        self.reader = AIMOInputReader(cfg)
        self.writer = AIMOOutputWriter(cfg)

    def run(self) -> None:
        
        with AIMOOutputPulse(self.cfg.output_ping_interval, self.cfg.output_pulse_enabled):
            self.run_with_pulse()

    def run_with_pulse(self) -> None:
        
        self.environment.configure()
        records = self.reader.read_records()
        results = []

        with AIMOServer(self.cfg):
            client = AIMOInferenceClient(self.cfg)
            solver = AIMOSolver(self.cfg, client)

            for record in records:
                result = solver.solve_problem(record)
                results.append(result)
                self.writer.write_results(results)

        self.writer.write_results(results)

In [ ]:
runner = AIMORunner(CFG)
runner.run()